In [ ]:
import sys, os
project_root = r"C:/Users/Asia/Desktop/mocadr/MoviesRecommendations"
sys.path.insert(0, project_root)

import pandas as pd
from modules.utils import create_cv_folds
import pandas as pd
import numpy as np
from sklearn.decomposition import NMF, TruncatedSVD
from modules.utils import build_rating_matrix, create_cv_folds, evaluate_fold
import torch
from rich.progress import Progress
import warnings
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import root_mean_squared_error
from modules.train import train_SVD2_model

df = pd.read_csv("C:/Users/Asia/Desktop/mocadr/MoviesRecommendations/data/ratings.csv")

folds = create_cv_folds(df, 5, "SVD2") # tu tworzymy foldy

rs = range(2, 11)
rmse = {}

with Progress() as p:
    t = p.add_task(description = "initialization", total=len(rs)*len(folds), visible=False)
    for r in rs: # tu sprawdzamy dla każdego r
        p.update(t, description=f"Training with r={r}", refresh=True, visible=True)
        r_rmse = []
        for fold in folds: # tu sprawdzamy po wszystkich foldach
            p.update(t, advance=1)
            Z_train = fold['Z_train']
            train_user_map = fold['user_map']
            train_movie_map = fold['movie_map']
            test_df = fold['test_df']

            Z_approx_train = train_SVD2_model(Z_train, r, return_obj = "Z") # tu dopasowujemy model na train

            # tu obliczamy i dodajemy rmse dla foldu
            r_rmse.append(evaluate_fold(test_df, train_user_map, train_movie_map, Z_approx_train))
      
        # tu dodajemy srednie rmse dla r
        rmse[r] = np.mean(r_rmse)

min_rmse = min(rmse.values())
best_r = list(rmse.keys())[list(rmse.values()).index(min_rmse)]
print(f"Best r = {best_r} with RMSE = {min_rmse:.4f}")

Output()

Best r = 6 with RMSE = 1.5310


In [ ]:
Z, user_map, movie_map = build_rating_matrix(df, "SVD2")
Z_imputed = train_SVD2_model(Z, 6, max_iter=1000)

In [5]:
print(Z_imputed)

(array([[ 3.3984287 ,  2.0919383 ,  1.664247  ,  1.3684602 , -1.5510484 ,
         0.5066682 ],
       [ 1.9665154 , -3.137074  , -0.3711144 , -1.3248806 ,  0.14010319,
        -0.13436215],
       [ 1.0490838 ,  0.7922255 ,  0.57376784,  1.3630323 ,  0.9341831 ,
         4.895638  ],
       ...,
       [ 3.7035196 , -0.92296195, -4.11995   ,  1.0821565 ,  0.74657273,
         1.4554157 ],
       [ 0.9453903 ,  0.16076821,  0.5638618 , -0.4330798 ,  2.6309516 ,
        -0.60051006],
       [ 4.271382  ,  1.2294946 , -4.140316  ,  0.3573777 ,  1.5530282 ,
         1.4840612 ]], shape=(610, 6), dtype=float32), array([[ 1.5142177 ,  1.2692068 ,  1.1026864 , ...,  0.09048936,
         0.09048936,  0.2991679 ],
       [-0.22145337, -0.17732765,  0.00966389, ..., -0.33396685,
        -0.33396685, -0.6815929 ],
       [ 0.59588027,  0.52291167,  0.43118697, ..., -0.32962877,
        -0.32962877, -0.19689608],
       [-0.06672901,  0.10067387,  0.14488305, ..., -0.28713375,
        -0.28713375

In [ ]:
from modules.train import train_NMF_model
Z_imputed = np.asarray(Z_imputed, dtype=float)
Z_imputed = np.maximum(Z_imputed, 0)
Z_approx = train_NMF_model(Z_imputed, 28)

ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (2,) + inhomogeneous part.